# Knowledge Distillation on Google Colab

This notebook provides an interactive interface for running knowledge distillation experiments on Google Colab.

**Hardware:** A100 GPU recommended

**Tasks:**
- BERT → DistilBERT on MNLI (target: 0.817 accuracy)
- BERT → DistilBERT on QQP (target: 0.900 accuracy)
- Llama 3B → 1B on translation (target: BLEU ~27, COMET ~0.82)

## Setup

In [ ]:
# Clone repository (if not already present)
import os

if not os.path.exists('Distillation'):
    !git clone https://github.com/YOUR_USERNAME/Distillation.git
    %cd Distillation
else:
    %cd Distillation

In [ ]:
# Install dependencies with uv (fast)
!pip install -q uv
!uv pip install -q -e .

# Or use traditional pip
# !pip install -q -r requirements.txt

In [ ]:
# Login to W&B
import wandb

wandb.login()

In [ ]:
# Login to HuggingFace (needed for Llama models)
from huggingface_hub import notebook_login

notebook_login()

## Check GPU

In [ ]:
!nvidia-smi

In [ ]:
# Verify environment detection
from src.utils.environment import detect_environment, get_device

print(f"Environment: {detect_environment()}")
print(f"Device: {get_device()}")

## Option 1: BERT → DistilBERT on MNLI

In [ ]:
# Run MNLI distillation
!python train.py \
--config
configs / bert_distilbert_mnli.yaml \
- -output_dir. / outputs / mnli_colab

## Option 2: BERT → DistilBERT on QQP

In [ ]:
# Run QQP distillation
!python train.py \
--config
configs / bert_distilbert_qqp.yaml \
- -output_dir. / outputs / qqp_colab

## Option 3: Llama 3B → 1B on Translation

In [ ]:
# Run translation distillation
!python train.py \
--config
configs / llama_3b_1b_translation.yaml \
- -output_dir. / outputs / translation_colab

## Evaluation

In [ ]:
# View final metrics
import json
from pathlib import Path

output_dir = "./outputs/mnli_colab"  # Change to your output directory
trainer_state = Path(output_dir) / "trainer_state.json"

if trainer_state.exists():
    with open(trainer_state) as f:
        state = json.load(f)

    print("Final metrics:")
    if 'best_metric' in state:
        print(f"Best metric: {state['best_metric']}")
    if 'log_history' in state:
        final_eval = [log for log in state['log_history'] if 'eval_loss' in log][-1]
        print(json.dumps(final_eval, indent=2))
else:
    print(f"No trainer state found at {trainer_state}")

## Download Model

In [ ]:
# Download trained model to local machine
from google.colab import files
import shutil

# Zip the output directory
output_dir = "./outputs/mnli_colab"  # Change to your output directory
shutil.make_archive("trained_model", 'zip', output_dir)

# Download
files.download("trained_model.zip")